# Fed CP 两序列 HTTP/1.1 传输修复

原冻结两次 HTTP/2 错误已保留；本修复只改协议，每URL补一次，来源/标签/费用不变。

In [ ]:
import hashlib,io,json,os,shlex,subprocess,tarfile,tempfile
from datetime import datetime,timezone
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _settings,_ssh,archive_nas_recorder,nas_api_call
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd().resolve();card=root/'research/fed_cp_rating_spread_source_20261023'
base=json.loads((card/'plan.json').read_text());repair=json.loads((card/'transport_repair_1_plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert repair['frozen'] and sha(card/'plan.json')==repair['original_plan_sha256']
assert sha(card/'retrieval_receipt.json')==repair['original_retrieval_receipt_sha256']
assert sha(card/'source_archive_receipt.json')==repair['original_source_archive_receipt_sha256']
old=json.loads((card/'retrieval_receipt.json').read_text());assert [r['curl_exit_code'] for r in old['responses']]==[92,92]
assert json.loads((card/'run_state.json').read_text())['status']=='transport_failed_no_mapping'
assert not (card/'repair1_run_state.json').exists()
assert sha(root/base['provider']['calendar_uri']/'qlib_store_manifest.json')==base['provider']['calendar_manifest_sha256']
state={'status':'HTTP1_1_transport_attempting','repair_plan_sha256':sha(card/'transport_repair_1_plan.json'),'original_HTTP_GET_attempts':2,'additional_HTTP_GET_attempts':0,'model_fits':0,'native_accounts':0}
atomic_json(state,card/'repair1_run_state.json')
snap=root/'data/snapshots/fed_cp_rating_spread_source_20261023/repair1';snap.mkdir(parents=True,exist_ok=True)
responses=[]
for req in base['requests']:
 series=req['series'];body=snap/(series+'.csv');headers=snap/(series+'.headers.txt')
 assert not body.exists() and not headers.exists()
 start=datetime.now(timezone.utc).isoformat();state['additional_HTTP_GET_attempts']+=1;atomic_json(state,card/'repair1_run_state.json')
 run=subprocess.run(['curl','--http1.1','-sS','--connect-timeout','5','--max-time','30','--max-redirs','0','-H','Accept: text/csv','-A','soxl-qlib-research/1.0','-D',str(headers),'-o',str(body),'-w','%{http_code}|%{url_effective}',req['url']],capture_output=True,text=True)
 end=datetime.now(timezone.utc).isoformat();parts=run.stdout.strip().split('|',1)
 if not body.exists():body.write_bytes(b'')
 if not headers.exists():headers.write_bytes(b'')
 responses.append({'series':series,'url':req['url'],'request_started_utc':start,'response_finished_utc':end,'http_status':int(parts[0]) if parts and parts[0].isdigit() else 0,'curl_exit_code':run.returncode,'curl_stderr':run.stderr[:2000],'final_url':parts[1] if len(parts)>1 else '','body_sha256':sha(body),'body_bytes':body.stat().st_size,'headers_sha256':sha(headers),'headers_bytes':headers.stat().st_size})
assert len(responses)==state['additional_HTTP_GET_attempts']==2
atomic_json({'status':'HTTP1_1_two_exact_requests_recorded','repair_plan_sha256':state['repair_plan_sha256'],'original_HTTP_GET_attempts':2,'additional_HTTP_GET_attempts':2,'responses':responses,'model_fits':0,'native_accounts':0},card/'transport_repair_1_retrieval.json')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='fed-cp-http1-') as temp:
 arc=Path(temp)/'source.tar.gz'
 with tarfile.open(arc,'w:gz') as tar:
  for req in base['requests']:
   for suffix in ('.csv','.headers.txt'):
    p=snap/(req['series']+suffix);tar.add(p,arcname=p.name)
 digest=sha(arc);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=arc,executor_id='soxl-minute-research')['artifact']
 assert receipt['sha256']==digest and receipt['size_bytes']==arc.stat().st_size
 remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='fed-cp-http1-restore-') as restore:
  p=Path(restore)/'source.tar.gz'
  with p.open('wb') as out:subprocess.run(['ssh','-o','BatchMode=yes','-p',str(settings['ssh']['port']),settings['ssh']['host'],'cat',remote_path],stdout=out,stderr=subprocess.PIPE,check=True)
  assert sha(p)==digest
  with tarfile.open(p,'r:gz') as tar:
   for req in base['requests']:
    for suffix in ('.csv','.headers.txt'):
     f=snap/(req['series']+suffix);raw=tar.extractfile(f.name).read();assert len(raw)==f.stat().st_size and hashlib.sha256(raw).hexdigest()==sha(f)
raw_receipt={'status':'HTTP1_1_exact_bytes_NAS_scoped_restore_verified','archive_sha256':digest,'archive_bytes':receipt['size_bytes'],'nas_path':remote_path,'evidence':receipt,'entries':[{'path':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(snap.iterdir()) if p.is_file()],'scoped_members_verified':4}
atomic_json(raw_receipt,card/'transport_repair_1_source_archive.json')
transport_ok=all(x['http_status']==200 and x['curl_exit_code']==0 and x['final_url']==x['url'] and 0<x['body_bytes']<=base['budget']['max_bytes_each'] for x in responses)
if not transport_ok:
 state['status']='HTTP1_1_transport_failed_no_mapping';atomic_json(state,card/'repair1_run_state.json')
 summary={'experiment':base['experiment'],'decision':'needs_exact_Fed_CP_source_transport_after_HTTP1_1_repair_no_fit','objective_achieved':False,'original_plan_sha256':repair['original_plan_sha256'],'repair_plan_sha256':state['repair_plan_sha256'],'original_HTTP_GET_attempts':2,'additional_HTTP_GET_attempts':2,'source_archive':'transport_repair_1_source_archive.json','transport_result':'transport_repair_1_retrieval.json','source_gate_passed':False,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
 atomic_json(summary,card/'summary.json')
 (card/'conclusion.md').write_text('# Fed商业票据等级利差：HTTP/1.1限定修复结果\n\n原两次HTTP/2请求均为curl92、未收到HTTP响应；原字节和NAS收据保留。按单独冻结修复计划以同URL各补一次HTTP/1.1，仍未取得两条有效原始CSV；精确状态和错误见transport_repair_1_retrieval.json。新响应/报头原字节也已NAS归档并限定恢复。0模型、0账户、0 SOXX结果。停止该传输版本，不换主机或日期继续尝试。\n')
 print(summary['decision'],[(x['series'],x['curl_exit_code'],x['http_status'],x['body_bytes']) for x in responses])
else:
 parsed={}
 for req in base['requests']:
  name=req['series'];d=pd.read_csv(io.BytesIO((snap/(name+'.csv')).read_bytes()),dtype=str)
  assert len(d.columns)==2 and d.columns[1]==name and d.columns[0] in ('DATE','observation_date')
  d.columns=['observation_date','value_string'];d['observation_date']=pd.to_datetime(d.observation_date,errors='raise').dt.normalize();d['value']=pd.to_numeric(d.value_string.replace('.',np.nan),errors='coerce')
  assert d.observation_date.is_monotonic_increasing and d.observation_date.is_unique and len(d)>3000
  parsed[name]=d[['observation_date','value']].rename(columns={'value':name})
 a,b=[r['series'] for r in base['requests']]
 paired=parsed[a].merge(parsed[b],on='observation_date',how='outer',validate='one_to_one').sort_values('observation_date')
 paired['spread_pctpoints']=paired[a]-paired[b];valid=paired.loc[np.isfinite(paired.spread_pctpoints)].copy();assert len(valid)>2500
 qlib.init(provider_uri=str(root/base['provider']['calendar_uri']),region='us')
 cal=pd.DatetimeIndex(D.calendar(start_time='2010-12-01',end_time=base['provider']['reused_held_signal_dates'][1],freq='day'))
 train=cal[(cal>=pd.Timestamp('2011-01-03'))&(cal<=pd.Timestamp('2022-12-30'))];held=cal[(cal>=pd.Timestamp('2023-01-03'))&(cal<=pd.Timestamp('2026-10-07'))]
 assert len(train)==3020 and len(held)==944
 vdates=valid.observation_date.to_numpy('datetime64[ns]');vspread=valid.spread_pctpoints.to_numpy(float);rows=[]
 for i,day in enumerate(cal):
  status='insufficient_lag';obs=pd.NaT;value=np.nan;age=np.nan
  if i>=5:
   j=int(np.searchsorted(vdates,np.datetime64(cal[i-5]),side='right')-1)
   if j>=0:
    obs=pd.Timestamp(vdates[j]);age=int(i-(cal.searchsorted(obs,side='right')-1))
    if age<=10:status='known_retrospective_not_vintage_qualified';value=float(vspread[j])
    else:status='stale_over_10_sessions'
   else:status='no_paired_observation'
  rows.append({'signal_date':day,'source_observation_date':obs,'source_age_sessions':age,'CP_A2P2_minus_AA_30day_pctpoints':value,'status':status})
 mapped=pd.DataFrame(rows);known=mapped.status.eq('known_retrospective_not_vintage_qualified');t=mapped.signal_date.isin(train);h=mapped.signal_date.isin(held);nt=int((known&t).sum());nh=int((known&h).sum());gate=nt/3020>=.95 and nh/944>=.95
 assert len(mapped)==len(cal)
 with R.start(experiment_name=base['experiment'],uri=str(card/'qlib_recorder')):
  rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=base,repair_plan=repair,source_archive=raw_receipt,original_csv_bytes={req['series']:(snap/(req['series']+'.csv')).read_bytes() for req in base['requests']});rec.save_objects(artifact_path='analysis',parsed=parsed,paired=paired,mapped=mapped);rec.log_metrics(model_fits=0,native_accounts=0,train_known_fraction=nt/3020,held_known_fraction=nh/944);rid,eid=rec.id,str(rec.experiment_id)
 binding={'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':base['experiment'],'source_plan':'plan.json','ssh':settings['ssh']};atomic_json(binding,card/'nas_binding.json')
 evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
 summary={'experiment':base['experiment'],'decision':'conditional_retrospective_Fed_CP_rating_spread_calendar_pass_first_vintage_pending_no_fit' if gate else 'needs_Fed_CP_pair_coverage_no_fit','objective_achieved':False,'original_plan_sha256':repair['original_plan_sha256'],'repair_plan_sha256':state['repair_plan_sha256'],'original_HTTP_GET_attempts':2,'additional_HTTP_GET_attempts':2,'source_archive':'transport_repair_1_source_archive.json','raw_rows_each':{k:len(v) for k,v in parsed.items()},'paired_finite_observations':len(valid),'train_calendar_days':3020,'train_known_days':nt,'held_calendar_days':944,'held_known_days':nh,'source_gate_passed':bool(gate),'historical_first_public_vintage_qualified':False,'model_fits':0,'native_accounts':0,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'nas_evidence':evidence,'nas_readback':{'sha256':remote,'bytes':evidence['size_bytes'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}}
 atomic_json(summary,card/'summary.json');state.update(status='completed',Qlib_D_calendar_calls=1,new_Qlib_Recorders=1,recorder_id=rid);atomic_json(state,card/'repair1_run_state.json')
 (card/'conclusion.md').write_text(f'# Fed 30日商业票据等级利差：HTTP/1.1来源结果\n\n原两次HTTP/2请求curl92、0响应字节，保持原NAS收据；一次预冻HTTP/1.1修复以同URL各接收一次。原始CSV和报头经哈希、NAS归档及限定恢复。按同日A2/P2减AA非金融30日利率，至少滞后5个真实Qlib会话，超过10会话未知；训练3020日已知{nt}，复用评价944日已知{nh}。决定`{summary["decision"]}`。当前回溯历史仍没有2011–2022原始发布/修订版本资格，未读SOXX收益、拟合或账户。Recorder `{rid}`，原始NAS SHA `{raw_receipt["archive_sha256"]}`，Recorder NAS SHA `{remote}`。\n')
 print(summary['decision'],nt,nh,len(valid),rid)


## 中文结论

结果以 conclusion.md 和 summary.json 为准，来源仍不等于策略。